<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Text_Embedding_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating a Text Embedding models

#### Refrence for this notebook is Hands-on-Large Language models

### Creating a Contrastive Learning model

In [9]:
from datasets import load_dataset

# load MNLI dataset from GLUE
# this dataset is based on ( entailment , contradiction , neutral)

train_dataset = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(100000))

train_dataset = train_dataset.remove_columns("idx")

lets look at dataset !!!

In [10]:
"""
    lets look at the structure of an datasets
    it contains 100000 rows having features: ['premise', 'hypothesis', 'label']

    here the term label have an 3 numbers from ( 0, 1, 2)
    0 = entailment  (both hypothesis and premises will be similar)
    1 = neutral     (both hypothesis and premises will be neutral)
    2 = contradiction (both hypothesis and premises will be opposite)
"""

train_dataset

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 100000
})

In [16]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

### Train an Embedding modal from scratch

In [17]:
from sentence_transformers import SentenceTransformer

# use a base modal
embedding_modal = SentenceTransformer("bert-base-uncased")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

now we need an loss function for sentence transformer , we will use `softmax` function for it .

In [19]:
from sentence_transformers import  losses

# define the loss function. in softmax loss, we will also needd to  set the number of labels

train_loss = losses.SoftmaxLoss(
    model = embedding_modal,
    sentence_embedding_dimension=embedding_modal.get_sentence_embedding_dimension(),
    num_labels=3
)

/tmp/ipykernel_1940/1839996282.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sentence_embedding_dimension=embedding_modal.get_sentence_embedding_dimension(),


Before training our model we need to create an evaluation metric to evaluate our modal ,

lets use Sementic Textual Similiraty Benchmark(STSB) , it is acollection of an human labeled dataset from sementic ranking form 1 to 5

we use this dataset to explore how our modal scores on this semantic simealrity tasks

In [22]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1 = val_sts["sentence1"],
    sentences2 = val_sts["sentence2"],
    scores = [score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)